# Databricks Connection Setup Guide

This notebook documents every step to connect your local environment to a Databricks workspace.

## Prerequisites
- Python 3.10-3.13 installed
- A Databricks workspace URL
- A Personal Access Token (PAT)
- A SQL Warehouse ID or Cluster ID

## Step 1: Install Databricks Connect

`databricks-connect` includes:
- `databricks-sdk` -- workspace client, SQL execution, jobs, clusters
- Spark Connect client -- run PySpark locally, execute on remote cluster

Use `--only-binary=:all:` to avoid C compiler issues on Windows.

In [ ]:
import sys
!{sys.executable} -m pip install databricks-connect --only-binary=:all: -q
print('Installation complete')

## Step 2: Create ~/.databrickscfg

Databricks tools automatically look for credentials in `~/.databrickscfg`.
This file lives in your home directory -- **never inside a git repo**.

```
[DEFAULT]
host = https://<your-workspace>.cloud.databricks.com
token = <your-personal-access-token>
warehouse_id = <your-sql-warehouse-id>
```

### How to get these values:

| Value | Where to find it |
|-------|------------------|
| host | Databricks workspace URL in your browser |
| token | Settings > Developer > Access tokens > Generate new token (choose API type) |
| warehouse_id | Compute > SQL Warehouses > click warehouse > copy ID from URL |
| cluster_id | Compute > All-purpose clusters > Configuration > Tags > ClusterId |

In [ ]:
import os

config_content = """[DEFAULT]
host = https://dbc-d8ec3837-9bcb.cloud.databricks.com
token = <YOUR_TOKEN_HERE>
warehouse_id = 716c425e63243341
"""

config_path = os.path.expanduser('~/.databrickscfg')
with open(config_path, 'w') as f:
    f.write(config_content)

print('Config written to:', config_path)

## Step 3: Verify Installation

In [ ]:
import importlib

for pkg in ['databricks.sdk', 'databricks.connect']:
    try:
        importlib.import_module(pkg)
        print(pkg, '-- OK')
    except ImportError as e:
        print(pkg, '-- MISSING:', e)

## Step 4: Connect to Workspace (Databricks SDK)

`WorkspaceClient()` reads credentials from `~/.databrickscfg` automatically.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print('Host        :', w.config.host)
print('Current user:', w.current_user.me().user_name)

## Step 6: Spark via Databricks Connect (Serverless)\n\nSince the workspace only has SQL Warehouses (no all-purpose clusters), we use **serverless compute**.\nDatabricks spins up compute automatically — no cluster ID needed.\n\n> Requires serverless compute to be enabled on your workspace (it is, since you have a Serverless Starter Warehouse).

In [ ]:
from databricks.connect import DatabricksSession\n\n# Serverless compute - no cluster ID needed\n# host + token are picked up from ~/.databrickscfg automatically\nspark = DatabricksSession.builder \\\n    .serverless(True) \\\n    .getOrCreate()\n\nprint('Spark version:', spark.version)

## Step 6: Spark via Databricks Connect (requires an All-Purpose Cluster)

> SQL Warehouses only support SQL. For PySpark / DataFrame API you need an **all-purpose cluster** running Databricks Runtime 13.0+.
>
> To create one: Databricks UI > Compute > Create compute > choose a runtime > start it > copy the Cluster ID.

In [ ]:
from databricks.connect import DatabricksSession

CLUSTER_ID = 'your-cluster-id-here'  # Replace with your all-purpose cluster ID

# host + token are picked up from ~/.databrickscfg automatically
spark = DatabricksSession.builder \
    .clusterId(CLUSTER_ID) \
    .getOrCreate()

print('Spark version:', spark.version)

In [ ]:
# Run a Spark SQL query -- executes on the remote cluster
df = spark.sql('SELECT current_user(), current_timestamp()')
df.show()

## Summary

| Step | What was done |
|------|---------------|
| 1 | Installed `databricks-connect` (includes SDK) with `--only-binary=:all:` |
| 2 | Created `~/.databrickscfg` with host, token, and warehouse_id |
| 3 | Verified packages imported correctly |
| 4 | Connected to workspace with `WorkspaceClient()` |
| 5 | Ran SQL on Serverless Starter Warehouse via `statement_execution` API |
| 6 | (Optional) Connected Spark session to an all-purpose cluster via Databricks Connect |

## Security Notes
- Never commit `~/.databrickscfg` or any file containing your token to git
- The `.gitignore` in this repo excludes `.databrickscfg` and `.env` files
- Rotate your PAT token regularly under Settings > Developer > Access tokens